# Göç ve İklim Verilerinin Birleştirilmesi

Bu notebook, `veri_temizleme.ipynb`'de temizlenen dosyaların birleştirilme sürecini içerir: iki bölgenin göç verisi tek tabloda toplanıyor, sıcaklık/yağış/afet verileriyle eşleştiriliyor ve son analiz tablosu oluşturuluyor.

In [1]:
import pandas as pd
import numpy as np

## İki bölgenin göç verisini birleştirme

Doğu-Güney Afrika ve Batı-Orta Afrika göç dosyaları aynı sütun yapısına sahip, alt alta ekliyoruz.

In [2]:
goc_dg = pd.read_excel("/content/dogu_guney_afrika_goc_temiz.xlsx")
goc_bo = pd.read_excel("/content/bati_orta_afrika_goc_temiz.xlsx")

goc = pd.concat([goc_dg, goc_bo], ignore_index=True)
print(goc.shape)
print(goc.duplicated(subset=["Year","Country of Asylum","Country of Origin"]).sum(), "duplicate")

(43555, 6)
0 duplicate


## Sıcaklık, yağış ve afet verileriyle eşleştirme

Eşleştirmeyi `Country of Origin ISO + Year` üzerinden yapıyoruz - yani göçün *nereden* kaynaklandığı ülkenin iklimine bakıyoruz, gidilen ülkeye değil. Önce eşleşme oranını kontrol ediyoruz.

In [3]:
sicaklik = pd.read_excel("/content/yillik_ortalama_sicaklik_temiz.xlsx")
yagis = pd.read_excel("/content/yillik_ortalama_yagis_temiz.xlsx")
afet_ozet = pd.read_excel("/content/afet_yil_ulke_ozet.xlsx")

test = goc.merge(sicaklik[["Year","Country ISO","Avg_Temp"]],
                  left_on=["Year","Country of Origin ISO"], right_on=["Year","Country ISO"], how="left")
print(test["Avg_Temp"].isna().sum(), "satır sıcaklıkla eşleşmedi")
print(sicaklik.duplicated(subset=["Year","Country ISO"]).sum(), "sıcaklıkta duplicate anahtar")

0 satır sıcaklıkla eşleşmedi
0 sıcaklıkta duplicate anahtar


Eşleşme tam (%100), duplicate anahtar yok - merge güvenli. Şimdi üç veriyi de birleştiriyoruz.

In [4]:
ana = goc.merge(sicaklik[["Year","Country ISO","Avg_Temp"]],
                 left_on=["Year","Country of Origin ISO"], right_on=["Year","Country ISO"], how="left") \
         .drop(columns=["Country ISO"])

ana = ana.merge(yagis[["Year","Country ISO","Avg_Precip"]],
                 left_on=["Year","Country of Origin ISO"], right_on=["Year","Country ISO"], how="left") \
         .drop(columns=["Country ISO"])

afet_cols = ["Year","ISO","Afet_Sayisi","Kuraklik_Sayisi","Sel_Sayisi","Firtina_Sayisi",
             "Asiri_Sicaklik_Sayisi","Toplam_Olum","Toplam_Etkilenen","Toplam_Hasar_1000USD"]
ana = ana.merge(afet_ozet[afet_cols],
                 left_on=["Year","Country of Origin ISO"], right_on=["Year","ISO"], how="left") \
         .drop(columns=["ISO"])

# bir ülke-yılda kayıtlı afet yoksa 0 yap
afet_sayisal = afet_cols[2:]
ana[afet_sayisal] = ana[afet_sayisal].fillna(0)

print(ana.shape)
ana.to_csv("ana_analiz_dosyasi.csv", index=False)

(43555, 16)


## Tekrar eden veri sorunu

`ana` tablosunda bir sorun var: `Avg_Temp`, `Avg_Precip`, `Afet_Sayisi` gibi sütunlar sadece Yıl+Origin'e bağlı, ama satırlar Yıl+Origin+Asylum bazında. Bu yüzden aynı iklim değeri, bir ülkenin gittiği her hedef ülke satırında tekrar tekrar yazılıyor.

In [5]:
tekrar = ana.groupby(["Year","Country of Origin"]).size()
print("Origin+Year başına ortalama satır (tekrar) sayısı:", round(tekrar.mean(),1))
print("En çok tekrar eden:", tekrar.idxmax(), "-", tekrar.max(), "satır")

ornek = ana[(ana["Country of Origin"]=="Somalia") & (ana["Year"]==2015)]
print(f"\nSomalia 2015: {len(ornek)} satır, hepsinde Avg_Temp aynı:", ornek["Avg_Temp"].nunique()==1)

Origin+Year başına ortalama satır (tekrar) sayısı: 36.6
En çok tekrar eden: (np.int64(2025), 'Sudan') - 110 satır

Somalia 2015: 104 satır, hepsinde Avg_Temp aynı: True


Bunu çözmek için tabloyu ikiye ayırıyoruz: göç akışları (Origin-Asylum bazında, tekrar yok) ve iklim/afet verisi (Origin-Year bazında, tekrar yok). Analiz sırasında ihtiyaç halinde tekrar birleştirilebilirler.

In [6]:
goc_tablosu = ana[["Year","Country of Asylum","Country of Origin",
                    "Country of Asylum ISO","Country of Origin ISO","Total"]].copy()

iklim_afet_tablosu = ana[["Year","Country of Origin","Country of Origin ISO",
                           "Avg_Temp","Avg_Precip","Afet_Sayisi","Kuraklik_Sayisi",
                           "Sel_Sayisi","Firtina_Sayisi","Asiri_Sicaklik_Sayisi",
                           "Toplam_Olum","Toplam_Etkilenen","Toplam_Hasar_1000USD"]].drop_duplicates().reset_index(drop=True)

print(goc_tablosu.shape, "-", goc_tablosu.duplicated().sum(), "duplicate")
print(iklim_afet_tablosu.shape, "-", iklim_afet_tablosu.duplicated(subset=["Year","Country of Origin ISO"]).sum(), "duplicate")

goc_tablosu.to_csv("goc_tablosu.csv", index=False)
iklim_afet_tablosu.to_csv("iklim_afet_tablosu.csv", index=False)

(43555, 6) - 0 duplicate
(1190, 13) - 0 duplicate


## İç göç satırlarının çıkarılması

`Country of Origin` ile `Country of Asylum` aynı olan satırlar (örn. Sudan'dan Sudan'a), sınır geçmeyen iç göçü/yerinden edilmeyi temsil ediyor. Analiz sınır ötesi göçe odaklandığı için bu satırları çıkarıyoruz.

In [7]:
print(goc_tablosu.shape)
ic_goc = (goc_tablosu["Country of Origin"] == goc_tablosu["Country of Asylum"]).sum()
print(ic_goc, "iç göç satırı bulundu")

goc_tablosu_temiz = goc_tablosu[goc_tablosu["Country of Origin"] != goc_tablosu["Country of Asylum"]].reset_index(drop=True)
print(goc_tablosu_temiz.shape)

goc_tablosu_temiz.to_csv("goc_tablosu_temiz.csv", index=False)

(43555, 6)
393 iç göç satırı bulundu
(43162, 6)


## Yıl + Ülke bazında toplulaştırılmış tablo

Hedef ülke (Asylum) detayına ihtiyaç duyulmayan analizler için, göç verisini de Yıl+Origin bazında toplayıp iklim/afet tablosuyla tek satırda birleştiriyoruz. Böylece hem tekrar kalmıyor hem de tek tabloda çalışmak mümkün oluyor.

In [8]:
goc_toplam = goc_tablosu_temiz.groupby(["Year","Country of Origin ISO"])["Total"].sum().reset_index()
goc_toplam.columns = ["Yil","ISO3","Goc"]

iklim_afet_tablosu2 = iklim_afet_tablosu.rename(columns={"Year":"Yil","Country of Origin ISO":"ISO3","Country of Origin":"Ulke"})

birlesik_tablo = iklim_afet_tablosu2.merge(goc_toplam, on=["Yil","ISO3"], how="left")

print(birlesik_tablo.shape)
print(birlesik_tablo.duplicated(subset=["Yil","ISO3"]).sum(), "duplicate")
print(birlesik_tablo.isna().sum())

birlesik_tablo.to_csv("birlesik_tablo.csv", index=False)
birlesik_tablo.head()

(1190, 14)
0 duplicate
Yil                      0
Ulke                     0
ISO3                     0
Avg_Temp                 0
Avg_Precip               0
Afet_Sayisi              0
Kuraklik_Sayisi          0
Sel_Sayisi               0
Firtina_Sayisi           0
Asiri_Sicaklik_Sayisi    0
Toplam_Olum              0
Toplam_Etkilenen         0
Toplam_Hasar_1000USD     0
Goc                      0
dtype: int64


,Yil,Ulke,ISO3,Avg_Temp,Avg_Precip,Afet_Sayisi,Kuraklik_Sayisi,Sel_Sayisi,Firtina_Sayisi,Asiri_Sicaklik_Sayisi,Toplam_Olum,Toplam_Etkilenen,Toplam_Hasar_1000USD,Goc
0,2001,Angola,AGO,21.84,1043.69,2.0,1.0,1.0,0.0,0.0,106.0,39928.0,0.0,480464
1,2002,Angola,AGO,21.95,1076.96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,449687
2,2003,Angola,AGO,22.01,1022.52,2.0,0.0,2.0,0.0,0.0,15.0,825.0,0.0,341284
3,2004,Angola,AGO,21.78,1059.90,4.0,1.0,3.0,0.0,0.0,28.0,358700.0,0.0,238382
4,2005,Angola,AGO,22.14,1011.95,1.0,0.0,1.0,0.0,0.0,0.0,10000.0,0.0,224147
